# Bird species classification with CNN, MobileNet and VGG16

Data: 525-species bird dataset (CC0), Hugging Face copy
chriamue/bird-species-dataset of the Kaggle gpiosenka/100-bird-species set.

In [ ]:
import os

CONFIG = {
    "hf_dataset": "chriamue/bird-species-dataset",
    "data_dir": "/content/birds" if os.path.isdir("/content") else "birds",
    "n_classes": None,   # None = all 525, or an int for a random subset
    "img_size": 224,
    "batch_size": 64,
    "seed": 42,          # species subset
    "train_seeds": [42, 43, 44],
    "augment": True,
    "mixed_precision": True,
    "models": ["cnn", "mobilenet", "vgg16", "vgg16_ft"],
    "epochs": {"cnn": 80, "mobilenet": 50, "vgg16": 40, "vgg16_ft": 40},
    "optimizer": {"cnn": ["adamax", 0.006], "mobilenet": ["adamax", 0.006],
                  "vgg16": ["sgd", 0.01], "vgg16_ft": ["sgd", 0.01]},
    "vgg_head": "original",   # or "compact"
    "label_smoothing": 0.1,
    "warmup_epochs": 2,
    "tta": True,              # average with horizontally flipped test image
    "early_stop_patience": 8,
    "lr_patience": 3,
    "lr_factor": 0.5,
}
OUT_DIR = "/content/drive/MyDrive/BM616_birds"

In [ ]:
import glob
import json
import random
import time

import numpy as np
import pandas as pd

# overrides for runs from the command line
CONFIG.update(json.loads(os.environ.get("BM616_CONFIG", "{}")))
OUT_DIR = os.environ.get("BM616_OUT_DIR", OUT_DIR)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    OUT_DIR = os.environ.get("BM616_OUT_DIR", os.path.abspath("BM616_birds"))

USE_DRIVE = IN_COLAB and os.environ.get("BM616_DRIVE", "1") != "0"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if IN_COLAB:
    # fonts for the figures
    os.system("echo ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula "
              "select true | debconf-set-selections")
    os.system("apt-get -qq install -y ttf-mscorefonts-installer fonts-stix > /dev/null")

import tensorflow as tf
import keras
from keras import layers

RESULTS_DIR = os.path.join(OUT_DIR, "results")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
FIG_DIR = os.path.join(OUT_DIR, "figures")
for d in (RESULTS_DIR, CKPT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

keras.utils.set_random_seed(CONFIG["seed"])
if CONFIG["mixed_precision"]:
    keras.mixed_precision.set_global_policy("mixed_float16")

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# download and unpack the data
import tarfile

from huggingface_hub import hf_hub_download

data_root = CONFIG["data_dir"]
for split in ("train", "valid", "test"):
    marker = os.path.join(data_root, f".{split}_unpacked")
    if os.path.exists(marker):
        continue
    archive = hf_hub_download(CONFIG["hf_dataset"], f"data/{split}.tar.gz",
                              repo_type="dataset")
    print("unpacking", split)
    with tarfile.open(archive) as tar:
        tar.extractall(data_root, filter="data")
    open(marker, "w").close()


def find_split_root(root):
    for dirpath, dirnames, _ in os.walk(root):
        names = {d.lower() for d in dirnames}
        if {"train", "valid", "test"} <= names:
            return dirpath
    raise FileNotFoundError(f"no train/valid/test folders under {root}")


split_root = find_split_root(data_root)
print("data in", split_root)


def norm(name):
    # some species names differ between splits by case or spacing
    return " ".join(name.upper().split())


def list_split(split):
    split_dir = next(os.path.join(split_root, d) for d in os.listdir(split_root)
                     if d.lower() == split)
    files = {}
    for cls in os.listdir(split_dir):
        cdir = os.path.join(split_dir, cls)
        if not os.path.isdir(cdir):
            continue
        imgs = sorted(glob.glob(os.path.join(cdir, "*")))
        imgs = [p for p in imgs if p.lower().endswith((".jpg", ".jpeg", ".png"))]
        files.setdefault(norm(cls), []).extend(imgs)
    return files


raw = {s: list_split(s) for s in ("train", "valid", "test")}
common = sorted(set(raw["train"]) & set(raw["valid"]) & set(raw["test"]))
dropped = sorted((set(raw["train"]) | set(raw["valid"]) | set(raw["test"])) - set(common))
if dropped:
    print("species missing from a split, dropped:", dropped)

if CONFIG["n_classes"]:
    rng = random.Random(CONFIG["seed"])
    common = sorted(rng.sample(common, CONFIG["n_classes"]))

CLASS_NAMES = common
NUM_CLASSES = len(CLASS_NAMES)
label_of = {c: i for i, c in enumerate(CLASS_NAMES)}


def items(split):
    return [(p, label_of[c]) for c in CLASS_NAMES for p in raw[split][c]]


TRAIN, VALID, TEST = items("train"), items("valid"), items("test")
print(f"{NUM_CLASSES} species | train {len(TRAIN)} | valid {len(VALID)} | test {len(TEST)}")

with open(os.path.join(RESULTS_DIR, "classes.json"), "w") as f:
    json.dump(CLASS_NAMES, f, indent=1)
counts = pd.DataFrame({
    "species": CLASS_NAMES,
    "train": [len(raw["train"][c]) for c in CLASS_NAMES],
    "valid": [len(raw["valid"][c]) for c in CLASS_NAMES],
    "test": [len(raw["test"][c]) for c in CLASS_NAMES],
})
counts.to_csv(os.path.join(RESULTS_DIR, "class_counts.csv"), index=False)
print(counts[["train", "valid", "test"]].describe().loc[["min", "50%", "max"]])

In [ ]:
# input pipeline; pixels stay in 0-255 and each model preprocesses itself
IMG = CONFIG["img_size"]
AUTOTUNE = tf.data.AUTOTUNE


def decode(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3,
                             expand_animations=False)
    img.set_shape([None, None, 3])
    return img, label


def resize(img, label):
    return tf.image.resize(img, [IMG, IMG]), label


def cached(split_items):
    paths = [p for p, _ in split_items]
    labels = [l for _, l in split_items]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    return ds.map(decode, num_parallel_calls=AUTOTUNE).cache()


def make_ds(base, seed=None):
    ds = base
    if seed is not None:   # shuffle only the training set
        ds = ds.shuffle(len(TRAIN), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(resize, num_parallel_calls=AUTOTUNE)
    return ds.batch(CONFIG["batch_size"]).prefetch(AUTOTUNE)


train_base = cached(TRAIN)
valid_ds = make_ds(cached(VALID))
test_ds = make_ds(cached(TEST))

In [ ]:
# models


def augmentation():
    # no vertical flip, test birds are never upside down
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.1),
    ], name="augment")


class VGGPreprocess(layers.Layer):
    # same as keras.applications.vgg16.preprocess_input

    def call(self, x):
        x = x[..., ::-1]
        return x - tf.constant([103.939, 116.779, 123.68], dtype=x.dtype)


def classifier(x):
    # float32 softmax for mixed precision
    return layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32",
                        name="predictions")(x)


def build_cnn():
    inp = keras.Input((IMG, IMG, 3))
    x = augmentation()(inp) if CONFIG["augment"] else inp
    x = layers.Rescaling(1 / 255.0)(x)
    for filters, n_conv in [(8, 2), (16, 2), (32, 2), (64, 2), (128, 2),
                            (256, 2), (512, 1)]:
        for _ in range(n_conv):
            x = layers.Conv2D(filters, 3, padding="same")(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.Flatten()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(512)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.5)(x)
    return keras.Model(inp, classifier(x), name="cnn")


def build_mobilenet():
    inp = keras.Input((IMG, IMG, 3))
    x = augmentation()(inp) if CONFIG["augment"] else inp
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)
    base = keras.applications.MobileNet(include_top=False, weights="imagenet",
                                        input_shape=(IMG, IMG, 3))
    x = base(x)
    x = layers.MaxPooling2D(x.shape[1])(x)
    x = layers.Flatten()(x)
    x = layers.BatchNormalization()(x)
    return keras.Model(inp, classifier(x), name="mobilenet")


def build_vgg16(fine_tune=False):
    inp = keras.Input((IMG, IMG, 3))
    x = augmentation()(inp) if CONFIG["augment"] else inp
    x = VGGPreprocess()(x)
    base = keras.applications.VGG16(include_top=False, weights="imagenet",
                                    input_shape=(IMG, IMG, 3))
    base.trainable = False
    if fine_tune:   # train block 5 only
        base.trainable = True
        for layer in base.layers:
            layer.trainable = layer.name.startswith("block5")
    x = base(x, training=False)
    if CONFIG["vgg_head"] == "original":
        x = layers.Flatten()(x)
        for units, drop in [(10000, 0.5), (10000, 0.4)]:
            x = layers.Dense(units)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
            x = layers.Dropout(drop)(x)
        x = layers.Dense(4000, activation="relu")(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Dense(700, activation="relu")(x)
    else:
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(1024)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Dropout(0.5)(x)
    return keras.Model(inp, classifier(x), name="vgg16")


BUILDERS = {"cnn": build_cnn, "mobilenet": build_mobilenet, "vgg16": build_vgg16,
            "vgg16_ft": lambda: build_vgg16(fine_tune=True)}


def run_id(name, seed):
    return f"{name}_s{seed}"


def smoothed_sparse_ce(eps):
    # keras has label smoothing only for one-hot labels
    def loss(y_true, y_pred):
        y = keras.ops.one_hot(keras.ops.cast(keras.ops.reshape(y_true, (-1,)),
                                             "int32"), NUM_CLASSES)
        return keras.losses.categorical_crossentropy(y, y_pred,
                                                     label_smoothing=eps)
    return loss


def compile_model(model, name):
    kind, lr = CONFIG["optimizer"][name]
    opt = {"adam": keras.optimizers.Adam, "adamax": keras.optimizers.Adamax,
           "sgd": keras.optimizers.SGD}[kind](lr)
    model.compile(
        optimizer=opt,
        loss=(smoothed_sparse_ce(CONFIG["label_smoothing"])
              if CONFIG["label_smoothing"] else "sparse_categorical_crossentropy"),
        metrics=["accuracy",
                 keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")],
    )
    return model

In [ ]:
# training


class Warmup(keras.callbacks.Callback):
    # linear lr ramp over the first few epochs

    def __init__(self, base_lr, epochs, steps_per_epoch):
        super().__init__()
        self.base_lr, self.total = base_lr, epochs * steps_per_epoch
        self.steps, self.epoch = steps_per_epoch, 0

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch = epoch

    def on_train_batch_begin(self, batch, logs=None):
        step = self.epoch * self.steps + batch + 1
        if step <= self.total:
            self.model.optimizer.learning_rate = self.base_lr * step / self.total


class EpochInfo(keras.callbacks.Callback):
    # log epoch time and lr

    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs["epoch_time_s"] = time.time() - self.t0
        logs["lr"] = float(keras.ops.convert_to_numpy(
            self.model.optimizer.learning_rate))


def train(name, seed):
    run = run_id(name, seed)
    done_flag = os.path.join(RESULTS_DIR, f"{run}_trained.json")
    if os.path.exists(done_flag):
        print(f"{run}: already trained, skipping")
        return
    backup = os.path.join(CKPT_DIR, f"{run}_backup")
    log_path = os.path.join(RESULTS_DIR, f"{run}_log.csv")
    resuming = os.path.isdir(backup) and os.listdir(backup)
    if not resuming and os.path.exists(log_path):
        os.remove(log_path)

    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    model = compile_model(BUILDERS[name](), name)
    print(f"{run}: {model.count_params() / 1e6:.2f}M parameters"
          + (" (resuming)" if resuming else ""))

    callbacks = [
        keras.callbacks.BackupAndRestore(backup),
        EpochInfo(),
        *([Warmup(CONFIG["optimizer"][name][1], CONFIG["warmup_epochs"],
                  -(-len(TRAIN) // CONFIG["batch_size"]))]
          if CONFIG["warmup_epochs"] else []),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss",
                                          factor=CONFIG["lr_factor"],
                                          patience=CONFIG["lr_patience"],
                                          min_lr=1e-6, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss",
                                      patience=CONFIG["early_stop_patience"],
                                      verbose=1),
        keras.callbacks.ModelCheckpoint(
            os.path.join(CKPT_DIR, f"{run}_best.weights.h5"),
            monitor="val_loss", save_best_only=True, save_weights_only=True),
        keras.callbacks.CSVLogger(log_path, append=True),
    ]
    model.fit(make_ds(train_base, seed), validation_data=valid_ds,
              epochs=CONFIG["epochs"][name], callbacks=callbacks,
              verbose=1 if USE_DRIVE else 2)

    trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    with open(done_flag, "w") as f:
        json.dump({"params_total": int(model.count_params()),
                   "params_trainable": trainable}, f)

In [ ]:
# evaluation on the test set
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_recall_fscore_support, top_k_accuracy_score)


def evaluate(name, seed):
    run = run_id(name, seed)
    if os.path.exists(os.path.join(RESULTS_DIR, f"{run}_eval.json")):
        return
    model = compile_model(BUILDERS[name](), name)
    model.load_weights(os.path.join(CKPT_DIR, f"{run}_best.weights.h5"))

    y_true = np.array([l for _, l in TEST])
    t0 = time.time()
    prob = model.predict(test_ds, verbose=0)
    ms_per_img = 1000 * (time.time() - t0) / len(y_true)
    acc_single = float(accuracy_score(y_true, prob.argmax(1)))
    if CONFIG["tta"]:
        flipped = test_ds.map(lambda x, y: (tf.image.flip_left_right(x), y))
        prob = (prob + model.predict(flipped, verbose=0)) / 2
    y_pred = prob.argmax(1)

    p, r, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=range(NUM_CLASSES), zero_division=0)
    pd.DataFrame({"species": CLASS_NAMES, "precision": p, "recall": r,
                  "f1": f1, "support": sup}).to_csv(
        os.path.join(RESULTS_DIR, f"{run}_per_class.csv"), index=False)
    np.save(os.path.join(RESULTS_DIR, f"{run}_confusion.npy"),
            confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES)))
    np.save(os.path.join(RESULTS_DIR, f"{run}_test_prob.npy"),
            prob.astype(np.float16))

    log = pd.read_csv(os.path.join(RESULTS_DIR, f"{run}_log.csv"))
    best = int(log["val_loss"].idxmin())
    with open(os.path.join(RESULTS_DIR, f"{run}_trained.json")) as f:
        params = json.load(f)
    result = {
        "model": name,
        "seed": seed,
        "n_classes": NUM_CLASSES,
        "n_test": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "accuracy_no_tta": acc_single,
        "top5_accuracy": float(top_k_accuracy_score(
            y_true, prob, k=5, labels=range(NUM_CLASSES))),
        "macro_precision": float(p.mean()),
        "macro_recall": float(r.mean()),
        "macro_f1": float(f1.mean()),
        "epochs_run": int(len(log)),
        "best_epoch": best + 1,
        "best_train_accuracy": float(log["accuracy"][best]),
        "best_val_accuracy": float(log["val_accuracy"][best]),
        "train_minutes": float(log["epoch_time_s"].sum() / 60),
        "test_ms_per_image": ms_per_img,
        **params,
        "config": CONFIG,
    }
    with open(os.path.join(RESULTS_DIR, f"{run}_eval.json"), "w") as f:
        json.dump(result, f, indent=1)
    print(f"{run}: acc {result['accuracy']:.4f} | macro F1 {result['macro_f1']:.4f}"
          f" | {result['epochs_run']} epochs, {result['train_minutes']:.0f} min")
    return result

In [ ]:
for seed in CONFIG["train_seeds"]:
    for name in CONFIG["models"]:
        train(name, seed)
        evaluate(name, seed)

summary = pd.DataFrame([
    json.load(open(os.path.join(RESULTS_DIR, f"{run_id(m, s)}_eval.json")))
    for s in CONFIG["train_seeds"] for m in CONFIG["models"]]).drop(columns="config")
print(summary.to_string())
print(summary.groupby("model")[["accuracy", "top5_accuracy", "macro_f1",
                                "epochs_run", "train_minutes"]].agg(["mean", "std"]))

In [ ]:
# Figures and tables from the saved results.

import glob
import json
import os
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from PIL import Image

# IEEE column and page width (inches)
COL_W = 3.5
PAGE_W = 7.16

# colour-blind safe colours
MODEL_COLORS = {"cnn": "#2a78d6", "mobilenet": "#eb6834", "vgg16": "#1baf7a",
                "vgg16_ft": "#8c5a00"}
MODEL_MARKERS = {"cnn": "o", "mobilenet": "s", "vgg16": "D", "vgg16_ft": "^"}
MODEL_LABELS = {"cnn": "CNN (scratch)", "mobilenet": "MobileNet",
                "vgg16": "VGG16 (frozen)", "vgg16_ft": "VGG16 (block 5 tuned)"}
TEXT = "#0b0b0b"
TEXT_2 = "#52514e"
GRID = "#e4e3df"
BLUE_RAMP = ["#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
             "#256abf", "#184f95", "#0d366b"]


FONT_DIRS = ["/usr/share/fonts/truetype/msttcorefonts",
             "/usr/share/fonts/truetype/stix", "/usr/share/fonts/opentype/stix"]


def register_fonts():
    # fonts installed after matplotlib started (Colab) are not in its cache
    for d in FONT_DIRS:
        for path in glob.glob(os.path.join(d, "*.[ot]tf")):
            font_manager.fontManager.addfont(path)
    names = {f.name for f in font_manager.fontManager.ttflist}
    if "Times New Roman" not in names:
        print("Times New Roman not found; falling back to STIX for text")


def set_style():
    register_fonts()
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "STIXGeneral", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "text.color": TEXT,
        "axes.labelcolor": TEXT,
        "axes.edgecolor": TEXT_2,
        "xtick.color": TEXT_2,
        "ytick.color": TEXT_2,
        "axes.linewidth": 0.6,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "grid.color": GRID,
        "grid.linewidth": 0.5,
        "lines.linewidth": 1.2,
        "legend.frameon": False,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def save(fig, fig_dir, name):
    os.makedirs(fig_dir, exist_ok=True)
    fig.savefig(os.path.join(fig_dir, name + ".pdf"))
    fig.savefig(os.path.join(fig_dir, name + ".png"))
    plt.close(fig)


def load_results(results_dir, models, seeds):
    # results[model] holds all seeds in 'runs' and the first seed at top level
    out = {}
    for m in models:
        runs = {}
        for s in seeds:
            pre = os.path.join(results_dir, f"{m}_s{s}")
            if not (os.path.exists(pre + "_log.csv")
                    and os.path.exists(pre + "_eval.json")):
                continue
            with open(pre + "_eval.json") as f:
                ev = json.load(f)
            runs[s] = {"log": pd.read_csv(pre + "_log.csv"), "eval": ev,
                       "per_class": pd.read_csv(pre + "_per_class.csv"),
                       "confusion": pre + "_confusion.npy"}
        if not runs:
            print(f"skipping {m}: results not found")
            continue
        first = next(iter(runs))
        out[m] = {"runs": runs, "seed": first, **runs[first]}
    return out


def seed_stats(result, key):
    # mean, min, max, sd over seeds
    v = np.array([r["eval"][key] for r in result["runs"].values()], dtype=float)
    return v.mean(), v.min(), v.max(), (v.std(ddof=1) if len(v) > 1 else 0.0)


def fig_samples(train_items, class_names, fig_dir, n=8, seed=0):
    rng = np.random.default_rng(seed)
    by_class = {}
    for path, label in train_items:
        by_class.setdefault(label, []).append(path)
    labels = rng.choice(sorted(by_class), size=n, replace=False)
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(COL_W, COL_W * rows / cols * 1.18))
    for ax, lab in zip(axes.flat, labels):
        path = rng.choice(by_class[lab])
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(wrap(class_names[lab]), fontsize=6, pad=2, linespacing=1.0)
        ax.axis("off")
    for ax in list(axes.flat)[len(labels):]:
        ax.axis("off")
    fig.subplots_adjust(wspace=0.05, hspace=0.25)
    save(fig, fig_dir, "samples")


def fig_class_counts(counts, fig_dir):
    counts = np.asarray(counts)
    fig, ax = plt.subplots(figsize=(COL_W, 1.7))
    bins = np.arange(counts.min() // 10 * 10, counts.max() + 10, 10)
    ax.hist(counts, bins=bins, color=MODEL_COLORS["cnn"], edgecolor="white",
            linewidth=0.6)
    med = np.median(counts)
    ax.axvline(med, color=TEXT_2, linewidth=0.8, linestyle=":")
    ax.text(med, ax.get_ylim()[1] * 0.95, f" median {med:.0f}", color=TEXT_2,
            fontsize=7, va="top")
    ax.set_xlabel("Training images per species")
    ax.set_ylabel("Number of species")
    save(fig, fig_dir, "class_counts")


def fig_curves(results, fig_dir):
    # first seed only
    models = list(results)
    fig, axes = plt.subplots(2, len(models), figsize=(PAGE_W, 3.1),
                             sharey="row", squeeze=False)
    for j, m in enumerate(models):
        log = results[m]["log"]
        ep = log["epoch"].to_numpy() + 1
        c = MODEL_COLORS[m]
        best = int(np.argmin(log["val_loss"].to_numpy()))
        lr = log["lr"].to_numpy() if "lr" in log else None
        drops = ep[1:][np.diff(lr) < 0] if lr is not None else []
        for i, (key, ylabel) in enumerate([("accuracy", "Accuracy"),
                                           ("loss", "Loss")]):
            ax = axes[i, j]
            for d in drops:
                ax.axvline(d - 0.5, color=GRID, linewidth=0.8, zorder=0)
            ax.plot(ep, log[key], color=c, linestyle="--", linewidth=1.0,
                    alpha=0.8, label="Training")
            ax.plot(ep, log["val_" + key], color=c, linewidth=1.4,
                    label="Validation")
            ax.plot(ep[best], log["val_" + key].iloc[best], marker="o",
                    markersize=4, color=c, markeredgecolor="white",
                    markeredgewidth=0.8, zorder=3)
            ax.set_xlim(0.5, ep[-1] + 0.5)
            if j == 0:
                ax.set_ylabel(ylabel)
            if i == 0:
                ax.set_title(MODEL_LABELS[m], color=TEXT)
                ax.set_ylim(top=1.0)
            else:
                ax.set_xlabel("Epoch")
                ax.set_ylim(bottom=0)
    axes[0, 0].legend(loc="lower right", handlelength=2.2)
    fig.tight_layout(w_pad=1.0, h_pad=0.6)
    save(fig, fig_dir, "training_curves")


def fig_metrics(results, fig_dir):
    # dot = mean over seeds, line = min to max
    metrics = [("accuracy", "Accuracy"), ("macro_precision", "Macro precision"),
               ("macro_recall", "Macro recall"), ("macro_f1", "Macro F1")]
    models = list(results)
    fig, ax = plt.subplots(figsize=(COL_W, 0.45 * len(models) + 0.9))
    ys = np.arange(len(metrics))[::-1]
    offs = np.linspace(0.27, -0.27, len(models))
    lo = 1.0
    for k, m in enumerate(models):
        stats = [seed_stats(results[m], key) for key, _ in metrics]
        lo = min(lo, min(st[1] for st in stats))
        y = ys + offs[k]
        ax.hlines(y, [st[1] for st in stats], [st[2] for st in stats],
                  color=MODEL_COLORS[m], linewidth=1.0, zorder=2)
        ax.scatter([st[0] for st in stats], y, s=20, color=MODEL_COLORS[m],
                   marker=MODEL_MARKERS[m], edgecolor="white", linewidth=0.6,
                   label=MODEL_LABELS[m], zorder=3)
        for st, yi in zip(stats, y):
            ax.text(st[2], yi, f"  {st[0]:.3f}", va="center", fontsize=6,
                    color=TEXT_2)
    ax.set_yticks(ys, [lab for _, lab in metrics])
    ax.set_xlim(np.floor(lo * 50) / 50 - 0.005, 1.012)
    ax.grid(axis="x")
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Score on test set")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2,
              handletextpad=0.2, columnspacing=1.0)
    save(fig, fig_dir, "test_metrics")


def fig_per_class_f1(results, fig_dir):
    fig, ax = plt.subplots(figsize=(COL_W, 1.9))
    for m in results:
        f1 = np.sort(np.mean([r["per_class"]["f1"].to_numpy()
                              for r in results[m]["runs"].values()], axis=0))
        x = np.arange(1, len(f1) + 1)
        ax.plot(x, f1, color=MODEL_COLORS[m], label=MODEL_LABELS[m])
    n = len(f1)
    ax.set_xlim(1, n)
    ax.set_ylim(top=1.01)
    ax.set_xlabel("Species, ranked by F1 score")
    ax.set_ylabel("Per-class F1 (mean of seeds)")
    ax.legend(loc="lower right")
    save(fig, fig_dir, "per_class_f1")


def fig_hardest(results, model, fig_dir, k=15):
    pc = results[model]["per_class"].sort_values("f1").head(k)[::-1]
    fig, ax = plt.subplots(figsize=(COL_W, 0.16 * k + 0.5))
    y = np.arange(len(pc))
    ax.barh(y, pc["f1"], height=0.7, color=MODEL_COLORS[model])
    for yi, v in zip(y, pc["f1"]):
        ax.text(v + 0.01, yi, f"{v:.2f}", va="center", fontsize=6, color=TEXT_2)
    ax.set_yticks(y, [pretty(s) for s in pc["species"]], fontsize=6)
    ax.set_xlim(0, 1.08)
    ax.grid(axis="x")
    ax.grid(axis="y", visible=False)
    ax.set_xlabel(f"F1 score ({MODEL_LABELS[model]})")
    save(fig, fig_dir, f"hardest_{model}")


def fig_confusion(cm, model, fig_dir):
    # with 525 classes an imshow is too fine to see, so plot only the errors
    cm = np.asarray(cm)
    n = cm.shape[0]
    off = cm.copy()
    np.fill_diagonal(off, 0)
    fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.92))
    ax.plot([0, n - 1], [0, n - 1], color=GRID, linewidth=1.2, zorder=1,
            label="Correct")
    shades = {1: (BLUE_RAMP[4], 14), 2: (BLUE_RAMP[6], 30), 3: (BLUE_RAMP[7], 48)}
    for count in sorted(set(off[off > 0].tolist())):
        colour, size = shades.get(count, (BLUE_RAMP[7], 48 + 12 * (count - 3)))
        t, p = np.nonzero(off == count)
        word = "image" if count == 1 else "images"
        ax.scatter(p, t, s=size, color=colour, edgecolor="white",
                   linewidth=0.6, zorder=3, label=f"{count} {word} wrong")
    ax.set_xlim(-5, n + 4)
    ax.set_ylim(n + 4, -5)
    ax.set_aspect("equal")
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_linewidth(0.4)
    ax.set_xlabel("Predicted species (index)")
    ax.set_ylabel("True species (index)")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3,
              frameon=False, handletextpad=0.4, columnspacing=1.2)
    save(fig, fig_dir, f"confusion_{model}")


def top_confusions(cm, class_names, k=6):
    cm = np.asarray(cm).copy()
    np.fill_diagonal(cm, 0)
    flat = np.argsort(cm, axis=None)[::-1][:k]
    pairs = []
    for idx in flat:
        t, p = np.unravel_index(idx, cm.shape)
        if cm[t, p] == 0:
            break
        pairs.append((int(t), int(p), int(cm[t, p])))
    return pairs


def fig_confused_pairs(cm, class_names, train_items, model, fig_dir, k=4, seed=0):
    pairs = top_confusions(cm, class_names, k)
    if not pairs:
        return
    rng = np.random.default_rng(seed)
    by_class = {}
    for path, label in train_items:
        by_class.setdefault(label, []).append(path)
    fig, axes = plt.subplots(2, len(pairs),
                             figsize=(PAGE_W * len(pairs) / 6, 2.6),
                             squeeze=False)
    for j, (t, p, n) in enumerate(pairs):
        for i, lab in enumerate([t, p]):
            ax = axes[i, j]
            ax.imshow(Image.open(rng.choice(by_class[lab])).convert("RGB"))
            ax.set_title(wrap(class_names[lab]), fontsize=6, pad=2,
                         linespacing=1.0)
            ax.grid(False)
            ax.set_xticks([])
            ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(False)
    axes[0, 0].set_ylabel("True species", fontsize=7)
    axes[1, 0].set_ylabel("Predicted as", fontsize=7)
    for j, (t, p, n) in enumerate(pairs):
        axes[1, j].text(0.5, -0.08, f"{n} test image{'s' if n > 1 else ''}",
                        transform=axes[1, j].transAxes, ha="center", va="top",
                        fontsize=6, color=TEXT_2)
    fig.subplots_adjust(wspace=0.08, hspace=0.45)
    save(fig, fig_dir, f"confused_pairs_{model}")


def write_tables(results, path):
    def cell(m, key, scale=1.0, fmt=".2f"):
        mean, _, _, sd = seed_stats(results[m], key)
        if len(results[m]["runs"]) == 1:
            return f"{scale * mean:{fmt}}"
        return f"{scale * mean:{fmt}} $\\pm$ {scale * sd:{fmt}}"

    lines = [
        r"\begin{tabular}{@{}lcccccc@{}}", r"\toprule",
        r"Model & Seeds & Params (M) & Epochs & Time (min) & Train acc. & Val. acc. \\",
        r"\midrule",
    ]
    for m, r in results.items():
        lines.append(
            f"{MODEL_LABELS[m]} & {len(r['runs'])} & "
            f"{r['eval']['params_trainable'] / 1e6:.1f} & "
            f"{cell(m, 'epochs_run', fmt='.0f')} & "
            f"{cell(m, 'train_minutes', fmt='.0f')} & "
            f"{cell(m, 'best_train_accuracy', 100)} & "
            f"{cell(m, 'best_val_accuracy', 100)} \\\\")
    lines += [r"\bottomrule", r"\end{tabular}", "", ""]
    lines += [
        r"\begin{tabular}{@{}lccccc@{}}", r"\toprule",
        r"Model & Accuracy & Top-5 & Precision & Recall & F1 \\",
        r"\midrule",
    ]
    for m in results:
        lines.append(
            f"{MODEL_LABELS[m]} & {cell(m, 'accuracy', 100)} & "
            f"{cell(m, 'top5_accuracy', 100)} & "
            f"{cell(m, 'macro_precision', 1, '.3f')} & "
            f"{cell(m, 'macro_recall', 1, '.3f')} & "
            f"{cell(m, 'macro_f1', 1, '.3f')} \\\\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")


def pretty(name):
    return " ".join(w.capitalize() for w in name.split())


def wrap(name, width=16):
    return "\n".join(textwrap.wrap(pretty(name), width))


def make_all(results_dir, fig_dir, models, seeds, class_names=None,
             train_items=None):
    set_style()
    results = load_results(results_dir, models, seeds)
    if not results:
        print("no results yet")
        return
    fig_curves(results, fig_dir)
    fig_metrics(results, fig_dir)
    fig_per_class_f1(results, fig_dir)
    best = max(results, key=lambda m: seed_stats(results[m], "macro_f1")[0])
    fig_hardest(results, best, fig_dir)
    cm = np.load(results[best]["confusion"])
    fig_confusion(cm, best, fig_dir)
    if class_names is not None and train_items is not None:
        fig_confused_pairs(cm, class_names, train_items, best, fig_dir)
    write_tables(results, os.path.join(results_dir, "tables.tex"))
    print("figures written to", fig_dir)

In [ ]:
# figures and tables
set_style()
fig_samples(TRAIN, CLASS_NAMES, FIG_DIR)
fig_class_counts(counts["train"], FIG_DIR)
make_all(RESULTS_DIR, FIG_DIR, CONFIG["models"], CONFIG["train_seeds"],
         CLASS_NAMES, TRAIN)

if USE_DRIVE:
    from IPython.display import Image as ShowImage, display
    for png in sorted(glob.glob(os.path.join(FIG_DIR, "*.png"))):
        print(os.path.basename(png))
        display(ShowImage(png, width=700))

In [ ]:
# zip results and figures
import zipfile

zip_path = os.path.join(OUT_DIR, "bm616_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ("results", "figures"):
        for path in glob.glob(os.path.join(OUT_DIR, sub, "*")):
            if not path.endswith("_test_prob.npy"):
                z.write(path, os.path.relpath(path, OUT_DIR))
print(zip_path)
if USE_DRIVE:
    from google.colab import files
    files.download(zip_path)